In [ ]:
import os
import pickle
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import RNN, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

from src.models import get_gru_variants
from src.evaluation import plot_roc_multiclass, plot_and_save_training_history

def set_seed(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
set_seed(42)

In [ ]:
# Change this number to 3, 5, or 10 (depending on R-Peaks)
NUM_PEAKS = 3
TARGET_LEN = 600

base_path = '/content/drive/MyDrive/File Tugas Kuliah/TA 3/'

print(f"Loading data untuk {NUM_PEAKS}R...")
with open(f'{base_path}X_train_{NUM_PEAKS}R.pkl', 'rb') as f:
    X_train = pickle.load(f)
with open(f'{base_path}y_train_{NUM_PEAKS}R.pkl', 'rb') as f:
    y_train = pickle.load(f)
with open(f'{base_path}X_test_{NUM_PEAKS}R.pkl', 'rb') as f:
    X_test = pickle.load(f)
with open(f'{base_path}y_test_{NUM_PEAKS}R.pkl', 'rb') as f:
    y_test = pickle.load(f)

le = LabelEncoder()
y_train_enc = to_categorical(le.fit_transform(y_train))
y_test_enc = to_categorical(le.transform(y_test))

print(f"Shape X_train: {X_train.shape}, y_train: {y_train_enc.shape}")
print(f"Shape X_test: {X_test.shape}, y_test: {y_test_enc.shape}")
print(f"Daftar Kelas: {le.classes_}")

In [ ]:
units = 128
dropout_rate = 0.3
learning_rate = 0.001
epochs = 50
batch_size = 64

gru_variants = get_gru_variants(units, dropout_rate)

trained_models = {}
histories = {}
output_dir = './hasil_evaluasi_gru'
os.makedirs(output_dir, exist_ok=True)

for name, gru_layer in gru_variants.items():
    print(f"\n{'='*40}")
    print(f"MEMULAI TRAINING: {name}")
    print(f"{'='*40}")

    model = Sequential([
        RNN(gru_layer, input_shape=(X_train.shape[1], X_train.shape[2])),
        BatchNormalization(),
        Dropout(dropout_rate),
        Dense(64, activation='relu'),
        Dense(len(le.classes_), activation='softmax')
    ])

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    history = model.fit(
        X_train, y_train_enc,
        validation_split=0.1,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[es],
        verbose=1
    )

    trained_models[name] = model
    histories[name] = history.history

    plot_and_save_training_history(history.history, name, output_dir)

In [ ]:
for name, model in trained_models.items():
    print(f"\n{'='*40}")
    print(f"EVALUASI MODEL: {name}")
    print(f"{'='*40}")

    y_pred_probs = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred_probs, axis=1)
    y_true_classes = np.argmax(y_test_enc, axis=1)

    target_names = [str(cls) for cls in le.classes_]
    print("\nClassification Report:")
    print(classification_report(y_true_classes, y_pred_classes, target_names=target_names))

    cm = confusion_matrix(y_true_classes, y_pred_classes)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
    fig, ax = plt.subplots(figsize=(8, 8))
    disp.plot(cmap=plt.cm.Blues, ax=ax)
    plt.title(f'Confusion Matrix - {name}')
    plt.show()

    plot_roc_multiclass(y_test_enc, y_pred_probs, model_name=name)